<a href="https://colab.research.google.com/github/BigAntSanchez77-hash/MLCapstone/blob/main/Capstone_Model_Experimentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎵 Music Genre Classification — Step 7: Experiment With Various Models

## Capstone: ML Engineering & AI Bootcamp

**Objective:** Build and compare multiple ML models for classifying music into 10 predefined genres using audio features extracted from the GTZAN dataset.

**Dataset:** [GTZAN Music Genre Dataset](https://huggingface.co/datasets/AntSanchez77/gtzan-music-genre-dataset) — 860 audio tracks (30s each), 10 genres, ~86 tracks per genre.

**Genres:** Blues, Classical, Country, Disco, Hip-Hop, Jazz, Metal, Pop, Reggae, Rock

---

### Pipeline Overview
1. **Data Loading** — Download GTZAN from Hugging Face
2. **Feature Extraction** — Extract audio features with librosa (MFCCs, spectral, rhythmic)
3. **Baseline Models** — Automated comparison of 7+ algorithms
4. **Performance Metrics** — Accuracy, F1 (macro), confusion matrices
5. **Hyperparameter Tuning** — GridSearchCV on top performers
6. **Cross-Validation** — Stratified K-Fold for robust evaluation
7. **Ensemble Methods** — VotingClassifier & StackingClassifier
8. **Overfitting Analysis** — Train vs. test performance gap
9. **Best Model Selection** — Final recommendation

## 1. Environment Setup & Dependencies

In [ ]:
# Install required packages (uncomment if needed)
# !pip install librosa datasets scikit-learn xgboost lightgbm matplotlib seaborn pandas numpy joblib tqdm

In [ ]:
import os
import warnings
import time
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa

from tqdm import tqdm
from datasets import load_dataset

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV,
    RandomizedSearchCV, cross_val_score, cross_validate
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, VotingClassifier, StackingClassifier,
    ExtraTreesClassifier
)
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('All imports loaded successfully.')

## 2. Load the GTZAN Dataset from Hugging Face

In [ ]:
# Load the GTZAN dataset from Hugging Face
# This downloads ~1.2 GB of audio data on first run
print('Loading GTZAN dataset from Hugging Face...')
print('(This may take a few minutes on first download)')

ds = load_dataset('AntSanchez77/gtzan-music-genre-dataset', split='train')

print(f'\nDataset loaded: {len(ds)} samples')
print(f'Features: {ds.features}')
print(f'\nColumn names: {ds.column_names}')

In [ ]:
# Quick look at a sample
sample = ds[0]
print('Sample keys:', list(sample.keys()))
print('Genre:', sample.get('genre', sample.get('label', 'unknown')))

if 'audio' in sample:
    audio_info = sample['audio']
    print(f'Audio sampling rate: {audio_info["sampling_rate"]} Hz')
    print(f'Audio array length: {len(audio_info["array"])}')
    print(f'Duration: {len(audio_info["array"]) / audio_info["sampling_rate"]:.1f} seconds')

## 3. Audio Feature Extraction

We extract a comprehensive set of audio features using **librosa**. These features capture timbral, spectral, and rhythmic properties that are well-suited for genre classification with traditional ML models.

| Feature Group | Features | Description |
|---|---|---|
| **Timbral** | 20 MFCCs (mean + std) | Mel-frequency cepstral coefficients — the gold standard for audio classification |
| **Spectral** | Centroid, Bandwidth, Rolloff, Contrast, Flatness | Shape of the frequency spectrum |
| **Rhythmic** | Tempo, Zero-Crossing Rate | Beat and temporal characteristics |
| **Harmonic** | Chroma (12 bins, mean + std), Tonnetz | Pitch class profiles |

In [ ]:
def extract_features(audio_array, sr=22050):
    """
    Extract a comprehensive set of audio features from a waveform.

    Returns a dictionary of ~80+ features covering timbral, spectral,
    rhythmic, and harmonic characteristics.
    """
    y = np.array(audio_array, dtype=np.float32)
    features = {}

    try:
        # --- MFCCs (20 coefficients) ---
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
        for i in range(20):
            features[f'mfcc_{i+1}_mean'] = np.mean(mfccs[i])
            features[f'mfcc_{i+1}_std'] = np.std(mfccs[i])

        # --- Delta MFCCs (first derivative, captures dynamics) ---
        delta_mfccs = librosa.feature.delta(mfccs)
        for i in range(20):
            features[f'delta_mfcc_{i+1}_mean'] = np.mean(delta_mfccs[i])

        # --- Spectral Features ---
        spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
        features['spectral_centroid_mean'] = np.mean(spectral_centroid)
        features['spectral_centroid_std'] = np.std(spectral_centroid)

        spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
        features['spectral_bandwidth_mean'] = np.mean(spectral_bandwidth)
        features['spectral_bandwidth_std'] = np.std(spectral_bandwidth)

        spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
        features['spectral_rolloff_mean'] = np.mean(spectral_rolloff)
        features['spectral_rolloff_std'] = np.std(spectral_rolloff)

        spectral_contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        for i in range(spectral_contrast.shape[0]):
            features[f'spectral_contrast_{i+1}_mean'] = np.mean(spectral_contrast[i])

        spectral_flatness = librosa.feature.spectral_flatness(y=y)[0]
        features['spectral_flatness_mean'] = np.mean(spectral_flatness)
        features['spectral_flatness_std'] = np.std(spectral_flatness)

        # --- Zero-Crossing Rate ---
        zcr = librosa.feature.zero_crossing_rate(y)[0]
        features['zcr_mean'] = np.mean(zcr)
        features['zcr_std'] = np.std(zcr)

        # --- RMS Energy ---
        rms = librosa.feature.rms(y=y)[0]
        features['rms_mean'] = np.mean(rms)
        features['rms_std'] = np.std(rms)

        # --- Tempo ---
        tempo = librosa.feature.tempo(y=y, sr=sr)
        features['tempo'] = tempo[0] if len(tempo) > 0 else 0.0

        # --- Chroma Features (12 pitch classes) ---
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        for i in range(12):
            features[f'chroma_{i+1}_mean'] = np.mean(chroma[i])
            features[f'chroma_{i+1}_std'] = np.std(chroma[i])

        # --- Tonnetz (tonal centroid) ---
        y_harmonic = librosa.effects.harmonic(y)
        tonnetz = librosa.feature.tonnetz(y=y_harmonic, sr=sr)
        for i in range(6):
            features[f'tonnetz_{i+1}_mean'] = np.mean(tonnetz[i])

    except Exception as e:
        print(f'  Error extracting features: {e}')
        return None

    return features

print(f'Feature extractor defined — extracts ~{len(extract_features(np.random.randn(22050*5), 22050))} features per track')

In [ ]:
# Extract features from all tracks
# This takes ~10-20 minutes depending on your hardware

FEATURES_CACHE = 'gtzan_features.csv'

if os.path.exists(FEATURES_CACHE):
    print(f'Loading cached features from {FEATURES_CACHE}...')
    df = pd.read_csv(FEATURES_CACHE)
    print(f'Loaded {len(df)} samples with {len(df.columns) - 1} features')
else:
    print('Extracting features from audio tracks...')
    records = []
    errors = []

    for idx in tqdm(range(len(ds)), desc='Extracting features'):
        sample = ds[idx]
        audio = sample['audio']
        y = np.array(audio['array'], dtype=np.float32)
        sr = audio['sampling_rate']

        # Get genre label
        genre = sample.get('genre', sample.get('label', None))

        feats = extract_features(y, sr)
        if feats is not None:
            feats['genre'] = genre
            records.append(feats)
        else:
            errors.append(idx)

    df = pd.DataFrame(records)
    df.to_csv(FEATURES_CACHE, index=False)
    print(f'\nExtraction complete: {len(df)} tracks processed, {len(errors)} errors')
    print(f'Features saved to {FEATURES_CACHE}')

print(f'\nDataset shape: {df.shape}')
print(f'\nGenre distribution:\n{df["genre"].value_counts()}')

In [ ]:
# Visualize genre distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Genre counts
genre_counts = df['genre'].value_counts()
genre_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('husl', 10))
axes[0].set_title('Samples per Genre')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Feature correlation heatmap (subset)
feature_cols = [c for c in df.columns if c != 'genre']
corr_subset = df[feature_cols[:20]].corr()
sns.heatmap(corr_subset, cmap='coolwarm', center=0, ax=axes[1],
            xticklabels=True, yticklabels=True, fmt='.1f')
axes[1].set_title('Feature Correlations (first 20 features)')

plt.tight_layout()
plt.show()

## 4. Data Preparation

In [ ]:
# Prepare features and labels
feature_cols = [c for c in df.columns if c != 'genre']
X = df[feature_cols].values
y_raw = df['genre'].values

# Encode labels
le = LabelEncoder()
y = le.fit_transform(y_raw)
genre_names = le.classes_
print(f'Genre labels: {list(genre_names)}')
print(f'Encoded as:   {list(range(len(genre_names)))}')

# Handle any NaN/Inf values
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

# Stratified train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'\nTrain set: {X_train.shape[0]} samples')
print(f'Test set:  {X_test.shape[0]} samples')
print(f'Features:  {X_train.shape[1]}')

## 5. Automated Model Comparison

We test **10 different classifiers** in an automated pipeline. Each model is wrapped with a `StandardScaler` to normalize features, and evaluated using **5-fold stratified cross-validation** on the training set, then evaluated on a held-out test set.

**Performance Metrics:**
- **Accuracy** — Overall correct predictions
- **Macro F1** — Harmonic mean of precision & recall, averaged equally across all 10 genres (important when we care about all genres equally)

In [ ]:
# Define all candidate models
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=2000, multi_class='multinomial', random_state=RANDOM_STATE
    ),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'SVM (RBF)': SVC(kernel='rbf', random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
    ),
    'Extra Trees': ExtraTreesClassifier(
        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, random_state=RANDOM_STATE
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, random_state=RANDOM_STATE,
        use_label_encoder=False, eval_metric='mlogloss', verbosity=0
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=200, random_state=RANDOM_STATE, verbose=-1
    ),
    'MLP Neural Net': MLPClassifier(
        hidden_layer_sizes=(256, 128, 64), max_iter=500,
        random_state=RANDOM_STATE, early_stopping=True
    ),
}

print(f'Testing {len(models)} models...')

In [ ]:
# Automated cross-validation comparison
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results = []

for name, model in models.items():
    print(f'\n{"="*60}')
    print(f'  Training: {name}')
    print(f'{"="*60}')

    # Build a pipeline with scaling
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])

    start_time = time.time()

    # Cross-validation on training set
    cv_results = cross_validate(
        pipe, X_train, y_train, cv=cv,
        scoring=['accuracy', 'f1_macro'],
        return_train_score=True,
        n_jobs=-1
    )

    # Also fit on full training set and evaluate on test set
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    elapsed = time.time() - start_time

    result = {
        'Model': name,
        'CV Accuracy (mean)': np.mean(cv_results['test_accuracy']),
        'CV Accuracy (std)': np.std(cv_results['test_accuracy']),
        'CV F1 Macro (mean)': np.mean(cv_results['test_f1_macro']),
        'CV F1 Macro (std)': np.std(cv_results['test_f1_macro']),
        'Train Accuracy': np.mean(cv_results['train_accuracy']),
        'Test Accuracy': accuracy_score(y_test, y_pred),
        'Test F1 Macro': f1_score(y_test, y_pred, average='macro'),
        'Training Time (s)': elapsed,
        'Overfit Gap': np.mean(cv_results['train_accuracy']) - np.mean(cv_results['test_accuracy'])
    }
    results.append(result)

    print(f'  CV Accuracy:  {result["CV Accuracy (mean)"]:.4f} ± {result["CV Accuracy (std)"]:.4f}')
    print(f'  CV F1 Macro:  {result["CV F1 Macro (mean)"]:.4f} ± {result["CV F1 Macro (std)"]:.4f}')
    print(f'  Test Accuracy: {result["Test Accuracy"]:.4f}')
    print(f'  Test F1 Macro: {result["Test F1 Macro"]:.4f}')
    print(f'  Overfit Gap:   {result["Overfit Gap"]:.4f}')
    print(f'  Time: {elapsed:.1f}s')

print(f'\n{"="*60}')
print('  All models complete!')
print(f'{"="*60}')

In [ ]:
# Results summary table
results_df = pd.DataFrame(results).sort_values('CV F1 Macro (mean)', ascending=False)
results_df = results_df.reset_index(drop=True)

# Display formatted table
display_cols = ['Model', 'CV Accuracy (mean)', 'CV F1 Macro (mean)',
                'Test Accuracy', 'Test F1 Macro', 'Overfit Gap', 'Training Time (s)']
print('\n📊 MODEL COMPARISON RESULTS (sorted by CV F1 Macro)\n')
print(results_df[display_cols].to_string(index=False, float_format='%.4f'))

In [ ]:
# Visualization: Model comparison
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

sorted_df = results_df.sort_values('CV F1 Macro (mean)', ascending=True)
colors = sns.color_palette('viridis', len(sorted_df))

# CV F1 Macro scores
axes[0].barh(sorted_df['Model'], sorted_df['CV F1 Macro (mean)'],
             xerr=sorted_df['CV F1 Macro (std)'], color=colors, capsize=3)
axes[0].set_xlabel('CV F1 Macro Score')
axes[0].set_title('Cross-Validation F1 Macro (5-Fold)')
axes[0].set_xlim(0, 1)

# Test Accuracy
axes[1].barh(sorted_df['Model'], sorted_df['Test Accuracy'], color=colors)
axes[1].set_xlabel('Test Accuracy')
axes[1].set_title('Test Set Accuracy')
axes[1].set_xlim(0, 1)

# Training time (log scale)
axes[2].barh(sorted_df['Model'], sorted_df['Training Time (s)'], color=colors)
axes[2].set_xlabel('Training Time (seconds)')
axes[2].set_title('Training Time per Model')
axes[2].set_xscale('log')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: model_comparison.png')